In [9]:
"""
Step 1 of 4 — build the forecast panel.

IN   WEOApr2026all.xlsx   IMF WEO Apr 2026, sheet "Countries", NGDP_RPCH
     OECD ... .csv        OECD EO 119, GDPV_ANNPCT
     AMECO6.TXT           AMECO OVGD, constant-price levels
OUT  panel.csv            iso, year, IMF, OECD, EC
"""
from google.colab import files
files.upload()

import os
import pandas as pd

YEARS = (2026, 2027)


def resolve(tokens, exts, label):
    """Match on name content, so '(1)' suffixes and Colab prefixes still work."""
    for f in sorted(os.listdir(".")):
        if f.lower().endswith(exts) and any(t in f.lower() for t in tokens):
            return f
    raise FileNotFoundError(
        f"no {label} file here. Wanted a name containing one of {tokens} "
        f"ending in {exts}.\nPresent: {sorted(os.listdir('.'))}")


F_WEO = resolve(("weo",), (".xlsx", ".xls"), "IMF WEO")
F_OECD = resolve(("gdpv_annpct", "oecd"), (".csv",), "OECD EO")
F_AMECO = resolve(("ameco",), (".txt", ".csv"), "AMECO")
print(f"IMF   {F_WEO}\nOECD  {F_OECD}\nEC    {F_AMECO}\n")

# --- IMF: growth is published directly -------------------------------------
imf = pd.read_excel(F_WEO, sheet_name="Countries")
imf = imf[imf["INDICATOR.ID"] == "NGDP_RPCH"][["COUNTRY.ID", *YEARS]]
imf = imf.rename(columns={"COUNTRY.ID": "iso"}).melt(
    "iso", var_name="year", value_name="IMF")
imf["IMF"] = pd.to_numeric(imf.IMF, errors="coerce")

# --- OECD: growth is published directly ------------------------------------
oecd = pd.read_csv(F_OECD)
oecd = oecd[oecd.TIME_PERIOD.isin(YEARS)][["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]]
oecd.columns = ["iso", "year", "OECD"]

# --- EC: AMECO gives LEVELS, so growth has to be computed ------------------
am = pd.read_csv(F_AMECO, sep=";", encoding="latin-1", low_memory=False)
am.columns = [c.strip() for c in am.columns]
am["CODE"] = am.CODE.astype(str).str.strip()
ov = am[am.CODE.str.endswith(".OVGD")].copy()
ov["iso"] = ov.CODE.str.split(".").str[0]
# AMECO codes Romania ROM, not ROU. Without this line the join silently drops
# Romania and the panel comes out 36 economies instead of 37.
ov["iso"] = ov.iso.replace({"ROM": "ROU"})

rows = []
for _, r in ov.iterrows():
    for y in YEARS:
        try:
            rows.append({"iso": r.iso, "year": y,
                         "EC": (float(r[str(y)]) / float(r[str(y - 1)]) - 1) * 100})
        except Exception:
            pass                      # aggregates and gap-years drop out here
ec = pd.DataFrame(rows)

# --- intersect: an economy must appear in all three, for both years --------
p = imf.merge(oecd, on=["iso", "year"]).merge(ec, on=["iso", "year"]).dropna()
p["year"] = p.year.astype(int)
n = p.groupby("iso").size()
p = p[p.iso.isin(n[n == len(YEARS)].index)]
p = p.sort_values(["iso", "year"]).reset_index(drop=True)
p.round(6).to_csv("panel.csv", index=False)

print(f"{p.iso.nunique()} economies x {len(YEARS)} years = {len(p)} cells "
      f"-> panel.csv")
print(", ".join(sorted(p.iso.unique())), "\n")

print("check against Table II          IMF    OECD      EC")
for iso in ("SWE", "DNK", "ITA"):
    for y in YEARS:
        r = p[(p.iso == iso) & (p.year == y)]
        if not r.empty:
            r = r.iloc[0]
            print(f"  {iso} {y}              {r.IMF:7.3f} {r.OECD:7.3f} {r.EC:7.3f}")

Saving AMECO6.TXT to AMECO6.TXT
Saving OECD.ECO.MAD,DSD_EO@DF_EO,+.GDPV_ANNPCT.A.csv to OECD.ECO.MAD,DSD_EO@DF_EO,+.GDPV_ANNPCT.A.csv
Saving WEOApr2026all.xlsx to WEOApr2026all.xlsx
IMF   WEOApr2026all.xlsx
OECD  OECD.ECO.MAD,DSD_EO@DF_EO,+.GDPV_ANNPCT.A.csv
EC    AMECO6.TXT

37 economies x 2 years = 74 cells -> panel.csv
AUS, AUT, BEL, BGR, CAN, CHE, CZE, DEU, DNK, ESP, EST, FIN, FRA, GBR, GRC, HRV, HUN, IRL, ISL, ITA, JPN, KOR, LTU, LUX, LVA, MEX, NLD, NOR, NZL, POL, PRT, ROU, SVK, SVN, SWE, TUR, USA 

check against Table II          IMF    OECD      EC
  SWE 2026                1.963   1.899   1.760
  SWE 2027                1.906   2.544   2.170
  DNK 2026                2.000   2.548   1.910
  DNK 2027                1.550   1.486   1.753
  ITA 2026                0.521   0.530   0.545
  ITA 2027                0.500   0.569   0.596
